# Human Landmarks with MediaPipe

> **Advanced · Applied vision**


## Why this matters

MediaPipe packages pretrained landmark models and tracking into a practical API. It is human landmark estimation—not `solvePnP` object pose—and has its own failure and privacy considerations.

**Where it appears:** Gesture controls, exercise visualizations, accessibility prototypes, and consent-based interactive installations.


## Learning Objectives

- Understand what MediaPipe adds on top of raw OpenCV (pretrained landmark models)
- Run hand/pose/face-mesh landmark detection with correct BGR/RGB handling
- Combine MediaPipe's landmarks with OpenCV drawing for a real pipeline


## Prerequisites

13 Video Processing and Background Motion; 03 OpenCV Setup and Your First Pipeline

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

MediaPipe Hands, Pose, Face Mesh; BGR↔RGB conversion; landmark coordinates

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### MediaPipe for Hands, Pose, and Face Mesh

MediaPipe provides highly-optimized, pretrained landmark models (hands: 21
points, pose: 33 points, face mesh: 468 points) that would otherwise
require training a custom model from scratch. OpenCV remains the
surrounding infrastructure: video capture, color conversion, and drawing.
A key gotcha: MediaPipe expects **RGB** input, while OpenCV loads/captures
in **BGR** -- forgetting this conversion silently degrades model accuracy
without raising any error.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Checking availability and the RGB/BGR handoff

MediaPipe is an optional dependency -- check for it explicitly and explain the required color-space conversion before calling any MediaPipe solution.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid, has_module


def to_mediapipe_rgb(bgr_image: np.ndarray) -> np.ndarray:
    """MediaPipe models expect RGB; OpenCV images are BGR. This one-line conversion is the
    single most common source of silently-degraded MediaPipe accuracy when skipped."""
    return cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)


print("MediaPipe available:", has_module("mediapipe"))
demo_frame = load_real_image("images/faces", "yoga.jpg")
rgb_frame = to_mediapipe_rgb(demo_frame)
print("BGR shape/dtype:", demo_frame.shape, demo_frame.dtype)
print(
    "RGB shape/dtype (same shape, channel ORDER differs):",
    rgb_frame.shape,
    rgb_frame.dtype,
)

### 2. Hand landmark detection

Run MediaPipe Hands on a frame and draw the 21 landmarks with OpenCV -- guarded so the notebook still explains the pattern even if MediaPipe isn't installed here.


In [ ]:
def ensure_mediapipe_models():
    """Download MediaPipe Tasks API models if they don't exist."""
    if not has_module("mediapipe"):
        return False
        
    import urllib.request
    from cv_utils import ensure_dir
    import os
    
    models_dir = os.path.join(get_real_data("models", ""), "")
    ensure_dir(models_dir)
    
    models = {
        "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        "pose_landmarker_full.task": "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
        "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    }
    
    for name, url in models.items():
        path = os.path.join(models_dir, name)
        if not os.path.exists(path):
            print(f"Downloading {name}...")
            urllib.request.urlretrieve(url, path)
            
    return True


def detect_hand_landmarks(bgr_image: np.ndarray):
    if not ensure_mediapipe_models():
        print("mediapipe unavailable -- skipping gracefully.")
        return None
        
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    model_path = get_real_data("models", "hand_landmarker.task")
    base_options = python.BaseOptions(model_asset_path=str(model_path))
    options = vision.HandLandmarkerOptions(base_options=base_options, num_hands=2)
    detector = vision.HandLandmarker.create_from_options(options)
    
    rgb = to_mediapipe_rgb(bgr_image)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    result = detector.detect(mp_image)
    return result


HAND_CONNECTIONS = [(0, 1), (1, 2), (2, 3), (3, 4), (0, 5), (5, 6), (6, 7), (7, 8), (5, 9), (9, 10), (10, 11), (11, 12), (9, 13), (13, 14), (14, 15), (15, 16), (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)]

def draw_hand_landmarks(bgr_image: np.ndarray, result) -> np.ndarray:
    if not ensure_mediapipe_models() or result is None:
        return bgr_image.copy()
        
    canvas = bgr_image.copy()
    h, w = canvas.shape[:2]
    
    if result.hand_landmarks:
        for hand_landmarks in result.hand_landmarks:
            # Draw connections
            for connection in HAND_CONNECTIONS:
                pt1 = hand_landmarks[connection[0]]
                pt2 = hand_landmarks[connection[1]]
                x1, y1 = int(pt1.x * w), int(pt1.y * h)
                x2, y2 = int(pt2.x * w), int(pt2.y * h)
                cv2.line(canvas, (x1, y1), (x2, y2), (255, 255, 255), 2)
                
            # Draw landmarks
            for landmark in hand_landmarks:
                x, y = int(landmark.x * w), int(landmark.y * h)
                cv2.circle(canvas, (x, y), 5, (0, 0, 255), -1)
                
    return canvas


hand_result = detect_hand_landmarks(demo_frame)
annotated = draw_hand_landmarks(demo_frame, hand_result)
show_grid(
    [
        ("input (synthetic -- no real hand present)", demo_frame),
        ("MediaPipe Hands output", annotated),
    ]
)


### 3. A unified wrapper for pose and face mesh

Generalize the pattern into one function covering MediaPipe's Hands/Pose/FaceMesh solutions consistently, rather than writing near-duplicate code for each.


In [ ]:
def run_mediapipe_solution(bgr_image: np.ndarray, solution_name: str):
    """solution_name in {'hands', 'pose', 'face_mesh'}. Returns the raw MediaPipe result, or
    None if mediapipe isn't installed."""
    if not ensure_mediapipe_models():
        return None
        
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    task_map = {
        "hands": ("hand_landmarker.task", vision.HandLandmarkerOptions, vision.HandLandmarker),
        "pose": ("pose_landmarker_full.task", vision.PoseLandmarkerOptions, vision.PoseLandmarker),
        "face_mesh": ("face_landmarker.task", vision.FaceLandmarkerOptions, vision.FaceLandmarker)
    }
    
    model_name, options_cls, detector_cls = task_map[solution_name]
    model_path = get_real_data("models", model_name)
    
    base_options = python.BaseOptions(model_asset_path=str(model_path))
    # Output face blendshapes for face_mesh to mimic old behavior if needed, but defaults are fine
    if solution_name == "face_mesh":
        options = options_cls(base_options=base_options, output_face_blendshapes=True)
    else:
        options = options_cls(base_options=base_options)
        
    detector = detector_cls.create_from_options(options)
    
    rgb = to_mediapipe_rgb(bgr_image)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    
    return detector.detect(mp_image)

for name in ("hands", "pose", "face_mesh"):
    result = run_mediapipe_solution(demo_frame, name)
    print(
        f"{name}: {'ran successfully' if result is not None else 'mediapipe unavailable'}"
    )

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — MediaPipe for Hands, Pose, and Face Mesh: Finger-Tracking Virtual Gesture Controller

By measuring the distance between landmark coordinates (such as the tip of the index finger and the thumb), we can simulate a volume knob or mouse cursor selector gesture control application.


In [ ]:
# Mock 3D coordinates for thumb and index fingertips (X, Y, Z coordinates)
thumb_tip = np.array([0.45, 0.60, -0.05])
index_tip = np.array([0.48, 0.40, -0.08])

# Calculate Euclidean distance between fingers
dist = np.linalg.norm(index_tip - thumb_tip)

# Map distance to volume percentage: assume 0.05 distance is closed, 0.25 is fully open
vol_pct = np.clip((dist - 0.05) / 0.20, 0.0, 1.0) * 100.0

print(f"Thumb Coordinate: {thumb_tip}")
print(f"Index Coordinate: {index_tip}")
print(f"Euclidean distance: {dist:.4f}")
print(f"Mapped Volume Level: {vol_pct:.1f}%")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — MediaPipe for Hands, Pose, and Face Mesh
1. Extend `draw_hand_landmarks` to also print the (x, y, z) coordinate of the index fingertip landmark (id 8) if a hand is detected.
2. Use MediaPipe Pose on a real webcam-captured frame (outside this notebook environment) and compare confidence to the synthetic-input case here.
3. Write a function estimating whether a detected hand is 'open' or 'closed' using landmark distances (fingertip-to-palm).

Use the empty cell below to work through them.


#### Solutions — MediaPipe for Hands, Pose, and Face Mesh

In [ ]:
# Solution 1: Print index fingertip coordinate
def print_index_fingertip(hand_landmarks) -> None:
    """Extract and display fingertip coordinates if available."""
    if hand_landmarks:
        # Index fingertip ID is 8
        idx_tip = hand_landmarks[8]
        print(
            f"Index Fingertip: X={idx_tip.x:.3f}, Y={idx_tip.y:.3f}, Z={idx_tip.z:.3f}"
        )


In [ ]:
# Solution 2: Webcam-captured frame comparison
# Explanation: In synthetic scenes, backgrounds are clean and contrast is high. Real webcam-captured
# frames contain varying lighting, shadow gradients, motion blur, and background clutter.
# This degrades MediaPipe's confidence scores, occasionally causing tracking dropouts.


In [ ]:
# Solution 3: Hand open/closed gesture classifier
def is_hand_open(hand_landmarks) -> bool:
    """Determine if hand is open by comparing fingertip-to-wrist distances."""
    # Wrist ID is 0. Fingertip IDs: Index=8, Middle=12, Ring=16, Pinky=20
    wrist = np.array([hand_landmarks[0].x, hand_landmarks[0].y])

    distances = []
    for tip_id in [8, 12, 16, 20]:
        tip = np.array(
            [hand_landmarks[tip_id].x, hand_landmarks[tip_id].y]
        )
        distances.append(np.linalg.norm(tip - wrist))

    # An open hand has larger average fingertip-to-wrist distance than a fist
    mean_dist = np.mean(distances)
    return mean_dist > 0.45


## Summary

You can run landmark estimation on images or video, map normalized landmarks to pixels, and build a small gesture interaction with graceful optional-dependency handling.

- **Best Practices:** Convert BGR to RGB exactly once at the model boundary, handle absent landmarks, test under varied poses and lighting, and protect camera/biometric data.
- **Common Pitfalls:** Confusing normalized and pixel coordinates, passing BGR to MediaPipe, and designing interactions that assume landmarks are always present.